# Faux Raccord · Session 3 — MCP

Project extension of **Step 8: A server for your project** from
`Agentic_04Nov_S3_Lab_Build_and_connect_an_MCP_server.ipynb`.

This notebook starts our Python MCP server over **stdio**, discovers
`search_passages` and calls the existing JavaScript BM25 search. The examples
use the **controlled Valkaama adaptation**, where the watch inconsistency was
inserted deliberately. They do not describe an error in the original screenplay.

Select the **Faux Raccord (Python 3.15.2)** kernel and run the cells in order.
The notebook needs Node.js and the project's Python environment; it does not
load `.env`, need an API key or call a language model.
The screenplay is by Tim Baumann, based on Hendrik Behnisch; the adapted text
and excerpts are CC BY-SA 3.0. Full attribution is recorded in the repository's
`data/controlled-script/license.json`.


In [1]:
import hashlib
import json
import sys
from contextlib import asynccontextmanager
from importlib.metadata import version
from pathlib import Path

from fastmcp import Client
from fastmcp.client.transports import StdioTransport


def find_project_root():
    # Works from the repository, agent/, or the original sibling course folder.
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        for candidate in (directory, directory / "faux-raccord"):
            if (candidate / "agent/server.py").is_file() and (candidate / "agent/tool_schemas.json").is_file():
                return candidate
    raise RuntimeError("Open this notebook from the Faux Raccord repository or its parent folder.")


ROOT = find_project_root()
assert sys.version_info[:2] == (3, 15), "Select the project's Python 3.15 kernel."
contracts = json.loads((ROOT / "agent/tool_schemas.json").read_text(encoding="utf-8"))
contract = next(tool for tool in contracts["tools"] if tool["name"] == "search_passages")


@asynccontextmanager
async def server_client():
    # A real log file works with Jupyter's stderr, which has no OS file descriptor.
    log_path = ROOT / ".local" / "mcp-notebook.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as log:
        transport = StdioTransport(
            command=sys.executable,
            args=[str(ROOT / "agent/server.py")],
            cwd=str(ROOT),
            env={"PYTHONUTF8": "1", "FASTMCP_LOG_LEVEL": "ERROR"},
            keep_alive=False,
            log_file=log,
        )
        async with Client(transport) as client:
            yield client


def fingerprint(description, parameters):
    approved_fields = {"description": description, "parameters": parameters}
    encoded = json.dumps(approved_fields, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()


EXPECTED_FINGERPRINT = fingerprint(contract["description"], contract["parameters"])


async def verify_contract(client):
    tools = await client.list_tools()
    assert len(tools) == 1, "Review a changed tool list before continuing."
    tool = tools[0]
    assert tool.name == contract["name"]
    assert tool.annotations is not None and tool.annotations.read_only_hint is True
    observed = fingerprint(tool.description, tool.input_schema)
    assert observed == EXPECTED_FINGERPRINT, "Tool description or schema changed; review it before use."
    return tool


def result_payload(result):
    assert not result.is_error, "The server returned a tool error."
    if result.structured_content is not None:
        return result.structured_content
    return json.loads("".join(getattr(item, "text", "") for item in result.content))


print(f"Python {sys.version.split()[0]} · FastMCP {version('fastmcp')}")
print("Transport: stdio · corpus: controlled-valkaama · model calls: 0")

Python 3.15.2 · FastMCP 4.1.6
Transport: stdio · corpus: controlled-valkaama · model calls: 0


## Discover the tool and check its contract

The client starts `server.py` as a subprocess and requests the tool list.
Its name, description and input schema must match the reviewed S2 contract;
the MCP annotation marks it as read-only. The fingerprint covers the description
and schema. This is a consistency check against our local contract, not a
certificate of a server's safety.

In [2]:
async with server_client() as client:
    tool = await verify_contract(client)
    print("Tool:", tool.name)
    print("Read-only annotation:", tool.annotations.read_only_hint)
    print("Description:", tool.description)
    print("Contract SHA-256:", EXPECTED_FINGERPRINT)
    print("Input schema:")
    print(json.dumps(tool.input_schema, indent=2, ensure_ascii=False))

Tool: search_passages
Read-only annotation: True
Description: Search the selected screenplay for a prop or event. Use short names or phrases to find evidence before drawing a conclusion. Returns up to five verbatim passages with scene identifiers and indicative page numbers. No matches does not prove that an explanation is absent.
Contract SHA-256: 63fc4d474d0f31b2412772fc89dae7a17701fbee20ec260765afd2671ea854c5
Input schema:
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "query": {
      "type": "string",
      "minLength": 1,
      "maxLength": 500,
      "pattern": "\\S",
      "description": "Prop name, alias or event to search for in the screenplay's language; whitespace alone is invalid."
    },
    "scene_ids": {
      "type": [
        "array",
        "null"
      ],
      "minItems": 1,
      "maxItems": 200,
      "uniqueItems": true,
      "items": {
        "type": "string",
        

## Call the real screenplay search

Use the example arguments from S2. The server forwards JSON to a fixed Node.js
bridge, which calls `retrieve()` from the application. The passages below come
from scenes **S001** and **S002** of the controlled adaptation. They should show
the broken watch and its later intact state.

Page numbers are indicative. BM25 scores rank lexical matches; they are not
probabilities that a continuity error exists.

In [3]:
arguments = contract["example_arguments"]
async with server_client() as client:
    await verify_contract(client)
    result = await client.call_tool("search_passages", arguments)
    search = result_payload(result)

assert search["script"]["id"] == "controlled-valkaama"
assert search["model_calls"] == 0
assert {passage["sceneId"] for passage in search["passages"]} == {"S001", "S002"}
assert 0 < len(search["passages"]) <= arguments["top_k"]

print(json.dumps({
    "script": search["script"]["title"],
    "license": search["script"]["license"],
    "revision": search["script"]["revision"],
    "query": search["query"],
    "scope": search["scope_scene_ids"],
    "passage_count": len(search["passages"]),
    "model_calls": search["model_calls"],
}, indent=2, ensure_ascii=False))

for passage in search["passages"]:
    print(f"\n[{passage['id']}] {passage['sceneId']} · page {passage['page']} · BM25 {passage['score']}")
    print(passage["text"])

{
  "script": "Valkaama — atelier de continuité",
  "license": "CC BY-SA 3.0",
  "revision": "bfb319fefc35664cdebef764bc6bf48ce22eaddf4127b5d4e385e4cfd1228407",
  "query": "brass pocket watch",
  "scope": [
    "S001",
    "S002"
  ],
  "passage_count": 2,
  "model_calls": 0
}

[S002-C1] S002 · page 3 · BM25 2.882
2) EXT. HALMSTAD - STREET - CAFE – DAY
Runa checks the same brass pocket watch: its glass is intact and its mechanism works. Nobody has repaired or replaced it.
Lasse opens the same red notebook, intact with all its original pages. No replacement has been obtained.
Ibsy pours water from her same yellow flask, now completely full. It has not been refilled.
Lasse unfolds his same folded street map. He has neither recovered it nor obtained a replacement.
Lasse repairs Ibsy's white toy car by firmly replacing its wheel. The repaired car rolls normally.
Runa, Lasse, Ibsy
Runa drinks a glass of red wine, Ibsy is busy with her cake
and Lasse is focused on the lines he is writing on 

## Distinguish an empty search from a rejected call

An unmatched query is a successful search with an empty passage list. It does
not establish that no explanation exists in the screenplay. An unknown scene
or `top_k` above five is an error: the caller must correct the request.

These checks exercise the real stdio connection. They do not assess an LLM's
quality or demonstrate an autonomous investigation loop.

In [4]:
async with server_client() as client:
    await verify_contract(client)
    empty = result_payload(await client.call_tool(
        "search_passages", {"query": "zzzxq_unmatched_screenplay_term_2026"}
    ))
    assert empty["passages"] == [] and empty["model_calls"] == 0
    print("Unmatched query: successful call, 0 passages, 0 model calls.")

    invalid_calls = [
        ("Unknown scene", {"query": "watch", "scene_ids": ["S999"]}, "UNKNOWN_SCENE"),
        ("Result limit exceeded", {"query": "watch", "top_k": 6}, "INVALID_ARGUMENTS"),
    ]
    for label, arguments, expected_code in invalid_calls:
        rejected = await client.call_tool("search_passages", arguments, raise_on_error=False)
        message = "".join(getattr(item, "text", "") for item in rejected.content)
        assert rejected.is_error and expected_code in message
        print(f"{label}: rejected — {message}")

print("MCP discovery, source search, empty-result handling and input checks completed.")

Unmatched query: successful call, 0 passages, 0 model calls.
Unknown scene: rejected — UNKNOWN_SCENE: Unknown scene identifiers: S999. Choose scenes from the selected screenplay.
Result limit exceeded: rejected — INVALID_ARGUMENTS: top_k: check maximum (5) against the tool schema.


MCP discovery, source search, empty-result handling and input checks completed.


## What this demonstrates

One read-only tool is exposed through MCP and returns real, referenced passages
from the selected screenplay. Its description and schema match S2, invalid
arguments return explicit errors, and the entire notebook uses **zero model
calls**. The other two S2 tools remain contracts for later implementation.